In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from spatialmath import SE2

from asrl.open3d_tests.env_builder import *
from asrl.slam.icp import icp
from asrl.slam.slam_gtsam import *

In [ ]:
og = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 1, 1],
    [0, 1, 1, 1, 0, 1, 1, 0],
    [0, 0, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0, 1, 0, 0],
    [0, 0, 1, 1, 1, 1, 0, 0],
])

m = ArrayMap(og, scale=1)

m.to_o3d_geometry()
scene = o3d.t.geometry.RaycastingScene()
walls = m.to_o3d_geometry()
for cube in walls:
    scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(cube))

In [ ]:
poses = np.array([
    [2.5, 1.5, 0],
    [2.5, 1.65, 0.8],
    [2.5, 1.8, 1.2],
    [2.5, 2.0, 1.5],
    [2.3, 2.3, 1.6],
    [2.3, 2.5, 1.6],
    [2.3, 2.8, 1.6],
    [2.3, 3.0, 1.6],
])

scans = collect_body_lidar_scans(poses, scene)

A = scans[0][:, :2]
B = scans[1][:, :2]
C = scans[2][:, :2]

T, _, _ = icp(A, B, max_dist=1.0)

A_new = (np.c_[A, np.ones(A.shape[0])] @ SE2(T).A.T)[:, :2]

plt.gca().set_aspect('equal')
# plt.scatter(A[:,0], A[:,1], c='r')
plt.scatter(B[:,0], B[:,1], c='b')
plt.scatter(A_new[:,0], A_new[:,1], c='g')

In [ ]:
slam = GraphICPSLAM2DGTSAM()
for scan in scans:
    slam.step(scan[:, :2])
estimated_poses = slam.poses()
print(np.linalg.det(slam.hessian()))

def plot_pose(a, color='r', length=0.25):
    dx = length * np.cos(a[2])
    dy = length * np.sin(a[2])
    plt.arrow(a[0], a[1], dx, dy, head_width=0.1, head_length=0.1, fc=color, ec=color)

def plot_poses(poses, color='r', length=0.05):
    for i in range(poses.shape[0]):
        plot_pose(poses[i, :], color=color, length=length)

plot_poses(poses, 'r')
plot_poses(estimated_poses, 'b')
